# VoxShield — anti-spoof training on KaggleKaggle instead of Colab. The differences that actually matter:| | Colab free | Kaggle free ||---|---|---|| GPU | T4 16 GB | **P100 16 GB** or **T4 x2** (32 GB total) || Quota | opaque, exhausts | **30 GPU-hours/week**, visible || Session | ~12 h | **12 h**, and `Save & Run All` runs with the tab closed || Storage | Drive, mounted | `/kaggle/working` **20 GB**, persists as notebook output || Google Drive | mountable | **not mountable — there is no equivalent** || Internet | on | **OFF by default — you must turn it on** |### Two things to do before running anything1. **Settings → Internet → On.** Needs phone verification on your account.   Without it `pip install` and the wav2vec2 download both fail.2. **Add Data** (top right) → search **`asvpoof-2019-dataset-la`** → Add.   That is ASVspoof 2019 LA, already on Kaggle — no download, no Drive, it   mounts read-only at `/kaggle/input/`.> **On mounting Drive:** you can't. Kaggle has no `google.colab.drive`. `gdown`> on a share link works for small files but fails on multi-GB ones (Drive's> virus-scan interstitial and quota errors). Use Kaggle Datasets instead —> which for ASVspoof means someone has already done the work for you.> **The DF archives on your Drive:** ASVspoof 2021 is also on Kaggle as> `mohammedabdeldayem/avsspoof-2021`. Attach that for the cross-condition> evaluation in section 8 rather than moving 34 GB off Drive.Nothing below hardcodes a dataset path. Kaggle re-packagers nest and renamethings freely, so the layout is **discovered** — splits are identified from theutterance ids (`LA_T_`, `LA_D_`, `LA_E_`, `DF_E_`), which are reliable in a wayfolder names are not.

## 1 · Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csvimport os, shutil, torchprint("\ntorch      ", torch.__version__)print("cuda       ", torch.cuda.is_available())if torch.cuda.is_available():    for i in range(torch.cuda.device_count()):        p = torch.cuda.get_device_properties(i)        print(f"  gpu {i}      {p.name}  {p.total_memory/1024**3:.1f} GB  sm_{p.major}{p.minor}")    print("bf16       ", torch.cuda.is_bf16_supported())print("cpus       ", os.cpu_count())total, used, free = shutil.disk_usage("/kaggle/working")print(f"disk       {free/1024**3:.0f} GB free")# Neither a P100 (sm_60) nor a T4 (sm_75) supports bf16, so the trainer's# resolve_precision() falls back to fp16 + GradScaler on its own.## If you were given T4 x2, this project uses ONE GPU - it does not shard. The# second card sits idle; that is fine and not worth the complexity here.

In [ ]:
# Internet check. Everything below needs it, and the failure is confusing if# you only find out three cells later.import sockettry:    socket.setdefaulttimeout(8)    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("pypi.org", 443))    print("internet: ON")except Exception as e:    raise SystemExit(        "internet: OFF - enable it in the right-hand panel under "        "Settings > Internet, then re-run. It requires phone verification."    )

## 2 · What did you attach?This walks `/kaggle/input` and reports every ASVspoof split it can find, byutterance-id prefix rather than by folder name.

In [ ]:
!ls /kaggle/input/

In [ ]:
import sys, subprocessfrom pathlib import PathWORK = Path("/kaggle/working/voxshield")if WORK.exists():    import shutil as _sh; _sh.rmtree(WORK)subprocess.run(["git", "clone", "-q", "-b", "karthik",                "https://github.com/SathvikGuttula/NullBox.git", str(WORK)], check=True)sys.path.insert(0, str(WORK / "backend"))%cd /kaggle/working/voxshield!git log --oneline -2

In [ ]:
!pip install -q "transformers>=4.44,<5" soundfileimport importlibfor m in ["torch", "torchaudio", "transformers", "soundfile", "librosa", "sklearn"]:    try:        print(f"{m:14}", importlib.import_module(m).__version__)    except Exception as e:        print(f"{m:14} MISSING  {type(e).__name__}")

In [ ]:
from app.ml import discoverylayout = discovery.discover("/kaggle/input")print(layout.describe())if not layout.audio:    raise SystemExit(        "No ASVspoof audio found under /kaggle/input.\n"        "Use 'Add Data' (top right) and attach 'asvpoof-2019-dataset-la'."    )if not layout.is_complete():    print("\n!! at least one split has audio but no protocol - see section 8 "          "for the DF keys, which ship separately")

## 3 · Manifests`--discover` uses the layout found above. Paths land as absolute`/kaggle/input/...`, which is correct — that mount is read-only and outside therepo.`train_heldout.csv` / `val_unseen.csv` hold attacks A05–A06 out of training.Train and dev share A01–A06, so dev EER saturates near zero within a few epochswhile eval-attack performance is still improving; selecting on dev picks amemorising checkpoint.

In [ ]:
!python backend/scripts/build_manifest.py \    --discover /kaggle/input \    --manifest-dir /kaggle/working/voxshield/datasets/manifests \    --holdout-attacks A05,A06

In [ ]:
!python backend/scripts/dataset_stats.py --all --check-audio --check-limit 1500

## 4 · Waveform cacheInto `/kaggle/temp`, **not** `/kaggle/working`. Working is capped at 20 GB andpersists as the notebook's output; an 8 GB rebuildable cache would eat 40 % ofthat quota for nothing. Temp is roomier and disposable — the cache costs a fewminutes to rebuild next session, and the checkpoint is what actually needs tosurvive.

In [ ]:
!mkdir -p /kaggle/temp/cache!python backend/scripts/cache_dataset.py \    --manifest datasets/manifests/train_heldout.csv \    --output /kaggle/temp/cache/train_heldout \    --max-seconds 6.0!python backend/scripts/cache_dataset.py \    --manifest datasets/manifests/val_unseen.csv \    --output /kaggle/temp/cache/val_unseen \    --max-seconds 6.0!du -sh /kaggle/temp/cache/* ; df -h /kaggle/temp | tail -1

## 5 · Size the run before committing GPU quota to itYou have 30 GPU-hours a week. `--dry-run` times a dozen real steps and printsprojected per-epoch and total wall clock plus peak VRAM, then stops — so amisjudged batch size costs thirty seconds, not three hours.

In [ ]:
!python backend/scripts/train_model.py \    --cache-dir /kaggle/temp/cache \    --train-manifest datasets/manifests/train_heldout.csv \    --validation-manifest datasets/manifests/val_unseen.csv \    --batch-size 32 --gradient-accumulation 1 \    --epochs 5 --freeze-epochs 1 --unfreeze-top-layers 6 \    --num-workers 2 \    --dry-run

## 6 · TrainCheckpoints go to `/kaggle/working`, which persists as this notebook's output.`--resume` picks up `experiments/kaggle/last.pt` — weights, AdamW moments, LRschedule and epoch counter — so a 12-hour cutoff costs one epoch.**Use `Save Version` → `Save & Run All (Commit)`** to run this with the tabclosed. Interactive sessions die when you disconnect; committed ones do not.

In [ ]:
!python backend/scripts/train_model.py \    --cache-dir /kaggle/temp/cache \    --train-manifest datasets/manifests/train_heldout.csv \    --validation-manifest datasets/manifests/val_unseen.csv \    --batch-size 32 --gradient-accumulation 1 \    --epochs 5 --freeze-epochs 1 --unfreeze-top-layers 6 \    --num-workers 2 \    --experiment-id kaggle \    --experiments-dir /kaggle/working/experiments \    --model-out /kaggle/working/models/voxshield_antispoof.pt \    --resume

In [ ]:
# What survives this session. Anything not under /kaggle/working is gone.!ls -lh /kaggle/working/models/ /kaggle/working/experiments/kaggle/ 2>/dev/null!du -sh /kaggle/working

### Continuing in a later session`/kaggle/working` is wiped when a session ends, but its contents are saved asthe version's **output**. To carry the checkpoint forward:1. **Save Version** on this notebook.2. In the new session: **Add Data → Your Work → Notebook Output → this notebook**.3. It mounts at `/kaggle/input/<notebook-slug>/`. Copy the checkpoint back and   point `--resume-from` at it:```python!mkdir -p /kaggle/working/experiments/kaggle!cp /kaggle/input/<notebook-slug>/experiments/kaggle/last.pt /kaggle/working/experiments/kaggle/```then add `--resume-from /kaggle/working/experiments/kaggle/last.pt` to thetraining cell. Raise `--epochs` too, or it will report that the schedule isalready finished and exit.

## 7 · Evaluate on ASVspoof 2019 LA eval71,237 utterances, attacks **A07–A19 — none seen in training**.

In [ ]:
!python backend/scripts/evaluate_model.py \    --manifest datasets/manifests/test.csv \    --model /kaggle/working/models/voxshield_antispoof.pt \    --calibrate-on datasets/manifests/validation.csv \    --batch-size 32 --num-workers 2 \    --save-scores \    --output-dir /kaggle/working/eval_la_eval

## 8 · Cross-condition evaluation on ASVspoof 2021 DFThe same model against unseen attacks **plus** codec degradation it nevertrained on. Expect the EER to be several times worse than on LA eval — that isthe honest result, and showing it next to the LA number is far more crediblethan showing only the good one.**Add Data → `mohammedabdeldayem/avsspoof-2021`** (or any DF mirror), then runthe cells below. DF labels ship separately from the audio; if the discoveryoutput says DF audio was found but no protocol, that is what is missing —`DF-keys-full.tar.gz` from asvspoof.org, which you already have on Drive andcan upload to Kaggle as a small private dataset.

In [ ]:
from app.ml import discoverydf_layout = discovery.discover("/kaggle/input")print(df_layout.describe())

In [ ]:
# Only run once DF audio AND a DF protocol are both present.!python backend/scripts/build_manifest.py \    --discover /kaggle/input \    --manifest-dir /kaggle/working/voxshield/datasets/manifests_df \    --holdout-attacks ""!python backend/scripts/evaluate_model.py \    --manifest /kaggle/working/voxshield/datasets/manifests_df/test.csv \    --model /kaggle/working/models/voxshield_antispoof.pt \    --batch-size 32 --num-workers 2 \    --save-scores \    --output-dir /kaggle/working/eval_df

If you evaluate DF in pieces (disk, or a session that ran out), pool theper-utterance scores rather than averaging the per-part EERs — EER is aproperty of the whole score distribution, and the average of four is adifferent, wrong number:```!python backend/scripts/merge_scores.py \    --scores /kaggle/working/eval_df_part*/scores.csv \    --output /kaggle/working/eval_df_full --label df```

## 9 · What to reportNever a single accuracy figure — these corpora are ~90 % spoof, so "always sayspoof" scores 90 % and flags every real customer.| | LA eval (A07–A19) | DF (pooled) ||---|---|---|| EER | | || ROC-AUC | | || normalised minDCF | | || miss rate @ 1 % false alarm | | |Both `report.json` files contain all four plus a per-attack breakdown.Two wordings that get checked:- **normalised minDCF**, not *t-DCF*. Tandem DCF folds in an ASV subsystem's  scores on the same trials; VoxShield has no enrolled ASV branch yet.- Always say which split a number came from. "EER 2.1 %" means nothing without  "on LA eval, attacks unseen in training".**If LA eval EER comes in under 0.5 %, be suspicious before you are pleased.**Re-read the `dataset_stats.py` leakage check and confirm you evaluated`test.csv` and not the split you trained on.